# 🧠 O'zbekcha Shaxsiy LLM (V3 Causal Transformer - 27.67M Parametr) — Google Colab GPU Treningi

Ushbu noutbuk noldan yaratilgan O'zbek tili Causal Transformer modelini Google Colab yuqori quvvatli serverlarida (NVIDIA Tesla T4 / A100 GPU) o'qitish va natijalarni kompyuteringizga ko'chirib olish uchun mo'ljallangan.

### 🌟 V3 Arxitektura Imkoniyatlari:
* **8,192 BPE Subword Lug'at**: O'zbek tili o'zaklari, qo'shimchalari va maxsus `<context>`, `</context>` tokenlari bilan boyitilgan;
* **27.67M Parametr (20M–30M Oralig'ida)**: 12 ta chuqur Transformer Decoder qatlami, 384 Hidden Dim, 12 ta Attention Head (head_dim=32), 1536 FFN, 256 kontekst oynasi;
* **2 Bosqichli O'qitish (Two-Stage Training)**:
  * **1-Bosqich (Pre-training)**: 20M–30M tokenlik xom o'zbek matnlarida tildagi umumiy grammatika va dunyoqarashni o'rganish;
  * **2-Bosqich (SFT / Instruct Tuning)**: Qat'iy mantiqiy savol-javoblar va Target Loss Masking orqali dialog uslubini egallash;
* **Universal RAG Context Injection**: Hujjatlar bazasidagi aniq faktlar model kontekstiga uzatiladi, bu esa gallyutsinatsiyani yo'qotadi;
* **100% Mahalliy Node.js / JavaScript Mosligi**: O'qitilgan `transformer_best.json` fayli hech qanday Python talab qilmasdan kompyuteringizda ishlaydi.

## 1-Qadam: Tizim muhitini tayyorlash, GPU tekshiruvi va kutubxonalarni o'rnatish

In [ ]:
# 1. GPU va apparatni tekshirish
!nvidia-smi

# 2. Node.js 20 LTS o'rnatish
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash -
!apt-get install -y nodejs

# 3. HuggingFace datasets kutubxonasi (Vikipediya korpusini yuklash uchun)
!pip install -q datasets torch

# 4. Versiyalarni tasdiqlash
!node -v
!python3 -c "import torch; print('PyTorch CUDA mavjud:', torch.cuda.is_available())"

## 2-Qadam: Loyiha paketini (`llm_colab_package.zip`) yuklash va ochish

In [ ]:
from google.colab import files
import os, zipfile

if not os.path.exists("package.json"):
    print("Iltimos, kompyuteringizdagi 'llm_colab_package.zip' faylini tanlang:")
    uploaded = files.upload()
    for fn in uploaded.keys():
        if fn.endswith('.zip'):
            with zipfile.ZipFile(fn, 'r') as zip_ref:
                zip_ref.extractall('.')
            print(f"✅ {fn} muvaffaqiyatli ochildi!")

!ls -la

## 3-Qadam: Korpus tayyorlash va 8,192 BPE Lug'atni o'qitish

In [ ]:
# 1. Katta o'zbek tili korpusini tayyorlash (onlayn yoki mahalliy bazalardan):
!python3 scripts/prepare_massive_uzbek_corpus.py

# 2. 8,192 BPE Subword Lug'atini o'qitish va eksport qilish:
!python3 scripts/train_bpe_8192.py --vocab_size=8192 --corpus=data/uzbek_pretrain_corpus.txt --output=data/bpe_vocab.json

# 3. RAG hujjatlar bazasini boyitish:
!node scripts/seed_rag_knowledge.js

## 4-Qadam: V3 Causal Transformer Modelini O'qitish (~27.67M Parametr GPU)

In [ ]:
# =========================================================================
# 🚀 1-BOSQICH (PRE-TRAINING): XOM KORPUSDA KEYINGI TOKENNI BASHORAT QILISH
# (27.67M parametr, 12 qatlam, 384 dim, 8192 vocab, 256 kontekst)
# =========================================================================
!python3 scripts/colab_train_gpu.py --version=v3 --stage=pretrain --epochs=3 --batch_size=32 --lr=0.0008

# =========================================================================
# 🎯 2-BOSQICH (SFT / INSTRUCT): DIALOG VA SAVOL-JAVOBLARDA O'QITISH
# (Target Loss Masking bilan modelga aniq javob berish madaniyati o'rgatiladi)
# =========================================================================
!python3 scripts/colab_train_gpu.py --version=v3 --stage=sft --epochs=20 --batch_size=32 --lr=0.00025

# =========================================================================
# ⚙️ TEZKOR VARIANT (V2 Modeli - ~2.59M parametr, atigi ~1.5 daqiqa):
# !python3 scripts/colab_train_gpu.py --version=v2 --stage=sft --epochs=25 --batch_size=32 --lr=0.0012
# =========================================================================

## 5-Qadam: 10 Ta Diverse Test Savollari Bo'yicha Sifatni Tekshirish (Evaluation)

In [ ]:
import subprocess

eval_script = '''
const fs = require("fs");
const path = require("path");
const CausalTransformerModel = require("./src/transformerModel");
const UzbekTokenizer = require("./src/uzbekTokenizer");
const DocumentRAGEngine = require("./src/rag");

const vocabPath = path.join(__dirname, "data", "bpe_vocab.json");
const ckptPath = path.join(__dirname, "data", "checkpoints", "transformer_best.json");

const tokenizer = new UzbekTokenizer();
tokenizer.loadFromFile(vocabPath);

const payload = JSON.parse(fs.readFileSync(ckptPath, "utf-8"));
const model = CausalTransformerModel.fromJSON(payload.modelData);
const rag = new DocumentRAGEngine();

const testQuestions = [
  "O'zbekiston Respublikasining rasmiy poytaxti qaysi shahar?",
  "Suv normal atmosfera bosimida necha gradusda qaynaydi?",
  "Sher bo'ridan kuchliroq. Bo'ri esa tulkidan kuchliroq. Bular orasida eng kuchlisi kim?",
  "150 dan 30 ni ayirib 4 ga bo'lsak nechiga teng bo'ladi?",
  "500 ning 20 foizi qancha bo'ladi?",
  "Ertalab o'z vaqtida uyg'onmay uxlab qolsang nima bo'ladi?",
  "Inson hayotida eng qimmatli boylik nima deb o'ylaysan?",
  "Fransiyaning poytaxti Toshkent shahri degan fikr to'g'rimi?",
  "Assalomu alaykum, do'stim! Kayfiyating qanday, charchamadingmi?",
  "12 ko'paytiruv 4 nechiga teng bo'ladi?"
];

console.log("=========================================================================");
console.log("  🎯 10 TA DIVERSE TEST SAVOLLARI BO'YICHA GENERATSIYA SIFATI AUDITI");
console.log("=========================================================================\\n");

testQuestions.forEach((q, idx) => {
  const ragRes = rag.retrieveContextForPrompt(q, 260);
  const prompt = ragRes.hasContext ? `${ragRes.contextPrompt}<human>: ${q}\\n<bot>:` : `<human>: ${q}\\n<bot>:`;
  const inputIds = tokenizer.encode(prompt);
  const genIds = model.generate(inputIds, 40, {
    temperature: 0.25,
    topK: 4,
    topP: 0.88,
    repetitionPenalty: 1.20
  });
  const decoded = tokenizer.decode(genIds, false);
  const parts = decoded.split(/<bot>:?/i);
  const reply = (parts.length > 1 ? parts[1] : decoded).replace(/<eos>|<human>.*/gi, "").trim();

  console.log(`[TEST ${idx + 1}/10]: "${q}"`);
  if (ragRes.hasContext) {
    console.log(`  📚 RAG Manbasi: [${ragRes.sources[0]?.title}]`);
  }
  console.log(`🤖 LLM Javobi: "${reply}"`);
  console.log("-------------------------------------------------------------------------");
});
'''

with open("eval_colab_10.js", "w", encoding="utf-8") as f:
    f.write(eval_script)

!node eval_colab_10.js


## 6-Qadam: O'qitilgan Modelni Kompyuterga Yuklab Olish (Download)

Ushbu katakchani ishga tushirsangiz, brauzeringiz orqali `transformer_best.json` va `bpe_vocab.json` fayllari kompyuteringizga avtomatik yuklanadi.
Yuklab olingan `transformer_best.json` faylini kompyuteringizdagi `data/checkpoints/` papkasiga qo'yishingiz kifoya!

In [ ]:
from google.colab import files
import os

ckpt = "data/checkpoints/transformer_best.json"
vocab = "data/bpe_vocab.json"

if os.path.exists(ckpt):
    print(f"📥 {ckpt} yuklab olinmoqda...")
    files.download(ckpt)
else:
    print(f"❌ {ckpt} topilmadi! Trening to'liq yakunlandimi?")

if os.path.exists(vocab):
    print(f"📥 {vocab} yuklab olinmoqda...")
    files.download(vocab)
